# Tome C3, Chapter 2: Grammars and Pushdown Automata (Python companion)

Exercises first, solutions at the end. We write grammars as Python dictionaries, parse them with Earley's algorithm (which accepts any grammar), convert them to Chomsky normal form and check that nothing was lost, count parse trees with the Cocke–Kasami–Younger table, simulate pushdown automata, and verify by brute force the more surprising claims of the chapter, such as the grammar for the complement of $\{ww\}$.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, random, time, math
from collections import defaultdict, deque
import matplotlib.pyplot as plt
random.seed(2)

def words(alphabet, n):
    for k in range(n + 1):
        for w in itertools.product(alphabet, repeat=k):
            yield ''.join(w)

## Exercises

**P2.1 (★) A general parser.** Represent a grammar as a dictionary from variables to lists of right-hand sides. Implement Earley's recogniser and test it on the grammars of Section 2.1 against the definitions of their languages (Proposition 2.3 for Dyck words).

**P2.2 (★★) Chomsky normal form.** Implement the four steps of Theorem 2.6 and check, with the Earley parser, that the converted grammar generates the same nonempty words as the original, on all words up to some length.

**P2.3 (★) Counting trees.** Implement the counting version of the CYK table (Exercise 2.5). Count the parse trees of $a+a+\cdots+a$ (Exercise 2.3), find ambiguous words in the grammar of Exercise 2.2, and compare the dangling-else grammar with ALGOL's (Exercise 2.4). Measure the cubic running time.

**P2.4 (★★) Pushdown automata.** Simulate nondeterministic PDAs by breadth-first search on configurations. Build the PDA of Theorem 2.9 from a grammar, and the palindrome PDA, and test them.

**P2.5 (★★) The complement of $\{ww\}$.** Check the grammar of Exercise 2.7 on all words up to length $12$.

## Solutions

### P2.1 A general parser

In [ ]:
def earley(G, start, w):
    # items (A, rhs, dot, origin); terminals are characters not in G
    chart = [set() for _ in range(len(w) + 1)]
    for rhs in G[start]: chart[0].add((start, rhs, 0, 0))
    for i in range(len(w) + 1):
        todo = list(chart[i])
        while todo:
            A, rhs, dot, org = todo.pop()
            if dot < len(rhs):
                X = rhs[dot]
                if X in G:                                   # predict
                    for r in G[X]:
                        it = (X, r, 0, i)
                        if it not in chart[i]: chart[i].add(it); todo.append(it)
                    # if X is nullable and already completed here, advance (Aycock-Horspool fix)
                    for (B, r2, d2, o2) in list(chart[i]):
                        if B == X and d2 == len(r2) and o2 == i:
                            it = (A, rhs, dot + 1, org)
                            if it not in chart[i]: chart[i].add(it); todo.append(it)
                elif i < len(w) and w[i] == X:               # scan
                    chart[i + 1].add((A, rhs, dot + 1, org))
            else:                                            # complete
                for (B, r2, d2, o2) in list(chart[org]):
                    if d2 < len(r2) and r2[d2] == A:
                        it = (B, r2, d2 + 1, o2)
                        if it not in chart[i]: chart[i].add(it); todo.append(it)
    return any(A == start and dot == len(rhs) and org == 0 for A, rhs, dot, org in chart[len(w)])

anbn = {'S': ['aSb', '']}
pal = {'S': ['aSa', 'bSb', 'a', 'b', '']}
dyck = {'S': ['(S)S', '']}
expr = {'E': ['E+T', 'T'], 'T': ['T*F', 'F'], 'F': ['(E)', 'a']}

def is_dyck(w):
    h = 0
    for c in w:
        h += 1 if c == '(' else -1
        if h < 0: return False
    return h == 0

checks = [("a^n b^n", anbn, 'ab', lambda w: w == 'a' * (len(w) // 2) + 'b' * (len(w) // 2) and len(w) % 2 == 0),
          ("palindromes", pal, 'ab', lambda w: w == w[::-1]),
          ("Dyck words", dyck, '()', is_dyck)]
for name, G, alph, truth in checks:
    ws = list(words(alph, 10))
    print(f"{name:12s}: Earley agrees with the definition on all {len(ws)} words of length <= 10:",
          all(earley(G, 'S', w) == truth(w) for w in ws))
print("expressions:", [(w, earley(expr, 'E', w)) for w in ["a+a*a", "(a+a)*a", "a+*a", "((a))", "a+(a"]])

Earley's algorithm (1970) works on any grammar, ambiguous or not, with $\varepsilon$-rules, without normal form; its worst case is cubic like CYK, but it is linear on many grammars used in practice. The small fix in the prediction step handles nullable variables correctly, a subtlety pointed out by Aycock and Horspool in 2002.

### P2.2 Chomsky normal form

In [ ]:
def to_cnf(G, start):
    G = {A: [list(r) for r in rs] for A, rs in G.items()}
    V = set(G)
    # 1. nullable variables, then remove epsilon-rules
    null, changed = set(), True
    while changed:
        changed = False
        for A, rs in G.items():
            if A not in null and any(all(X in null for X in r) for r in rs):
                null.add(A); changed = True
    H = {}
    for A, rs in G.items():
        new = set()
        for r in rs:
            opts = [[X] if X not in null else [X, None] for X in r]
            for choice in itertools.product(*opts):
                rr = tuple(X for X in choice if X is not None)
                if rr: new.add(rr)
        H[A] = new
    # 2. unit rules
    unit = {A: {A} for A in H}
    changed = True
    while changed:
        changed = False
        for A in H:
            for B in list(unit[A]):
                for r in H[B]:
                    if len(r) == 1 and r[0] in V and r[0] not in unit[A]:
                        unit[A].add(r[0]); changed = True
    H = {A: {r for B in unit[A] for r in H[B] if not (len(r) == 1 and r[0] in V)} for A in H}
    # 3. long rules and 4. terminals
    out, fresh = defaultdict(set), itertools.count()
    term = {}
    def T(a):
        if a not in term:
            term[a] = f"T_{a}"; out[term[a]].add((a,))
        return term[a]
    for A, rs in H.items():
        for r in rs:
            if len(r) == 1: out[A].add(r); continue
            r = [X if X in V else T(X) for X in r]
            cur = A
            while len(r) > 2:
                nxt = f"{A}_{next(fresh)}"
                out[cur].add((r[0], nxt)); cur, r = nxt, r[1:]
            out[cur].add(tuple(r))
    return dict(out)

def cyk(G, start, w, count=False):
    n = len(w)
    if n == 0: return 0
    binary = [(A, r[0], r[1]) for A, rs in G.items() for r in rs if len(r) == 2]
    N = defaultdict(int)
    for i, c in enumerate(w):
        for A, rs in G.items():
            if (c,) in rs: N[i, i, A] += 1
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            for A, B, Cc in binary:
                s = 0
                for k in range(i, j):
                    b = N.get((i, k, B), 0)
                    if b:
                        s += b * N.get((k + 1, j, Cc), 0)
                if s: N[i, j, A] += s
    return N.get((0, n - 1, start), 0)

for name, G, S, alph, L in [("palindromes", pal, 'S', 'ab', 9), ("Dyck", dyck, 'S', '()', 12),
                            ("expressions", expr, 'E', 'a+*()', 6), ("equal a,b", {'S': ['aSbS', 'bSaS', '']}, 'S', 'ab', 10)]:
    H = to_cnf(G, S)
    nrules = sum(len(r) for r in H.values())
    same = all((cyk(H, S, w) > 0) == earley(G, S, w) for w in words(alph, L) if w)
    print(f"{name:12s}: CNF with {len(H):2d} variables and {nrules:3d} rules; same nonempty words up to length {L}: {same}")
print("CNF of the expression grammar:")
for A, rs in sorted(to_cnf(expr, 'E').items()):
    print(f"  {A:6s} -> " + " | ".join(' '.join(r) for r in sorted(rs)))

The normal form of the expression grammar is already larger and much less readable than the original; unit rules like $E\to T$ have been folded into $E$, which now has all the rules of $T$ and $F$. The checks compare two independent parsers on every short word, so a mistake in either the conversion or the parsers would very likely show up.

### P2.3 Counting trees

In [ ]:
amb = to_cnf({'E': ['E+E', 'a']}, 'E')
catalan = lambda n: math.comb(2 * n, n) // (n + 1)
print("parse trees of a(+a)^n:", [cyk(amb, 'E', 'a' + '+a' * n) for n in range(10)])
print("Catalan numbers:       ", [catalan(n) for n in range(10)])

eq = to_cnf({'S': ['aSbS', 'bSaS', '']}, 'S')
print("equal a,b grammar, trees of some words:", {w: cyk(eq, 'S', w) for w in ['ab', 'abab', 'aabb', 'ababab', 'abba' * 2]})

dangling = to_cnf({'S': ['iS', 'iSeS', 'x']}, 'S')
algol = to_cnf({'S': ['U', 'iU', 'iUeS'], 'U': ['x', 'bSd']}, 'S')
print("dangling else:", {w: cyk(dangling, 'S', w) for w in ['iixex', 'iiixexex', 'iiiixexex']})
print("ALGOL 60 style:", {w: cyk(algol, 'S', w) for w in ['ibixdex', 'ibixexd', 'ibibixdexdex']})
worst = max((cyk(algol, 'S', w), w) for w in words('ibdxe', 7) if w)
print("largest number of ALGOL parse trees over all words of length <= 7:", worst[0])

In [ ]:
G = to_cnf(dyck, 'S')
ns, ts = [20, 40, 60, 80, 100, 120], []
for n in ns:
    w = '(' * (n // 2) + ')' * (n // 2)
    t = time.perf_counter(); cyk(G, 'S', w); ts.append(time.perf_counter() - t)
plt.figure(figsize=(5, 3))
plt.loglog(ns, ts, "o-", label="CYK")
plt.loglog(ns, [ts[-1] * (n / ns[-1]) ** 3 for n in ns], "--", label="slope 3")
plt.xlabel("n"); plt.ylabel("seconds"); plt.legend(); plt.tight_layout(); plt.show()
print("time ratio for doubling n from 60 to 120:", round(ts[-1] / ts[2], 1))

The counts of $a(+a)^n$ are the Catalan numbers, as Exercise 2.3 proves. The grammar $S\to aSbS\mid bSaS\mid\varepsilon$ is ambiguous already on $abab$, and the number of trees grows quickly with the length. The dangling-else grammar gives two trees to $\texttt{iixex}$, as in Exercise 2.4, and more for longer words; the ALGOL grammar never gives more than one, on every word up to length $7$. On the log-log plot the running time of CYK follows a line of slope $3$: doubling the length multiplies the time by about $8$.

### P2.4 Pushdown automata

In [ ]:
def pda_accepts(moves, start, accept, w, max_stack=None):
    # moves: list of (state, read, pop, new_state, push); read/pop/push are '' or one symbol
    max_stack = max_stack or 2 * len(w) + 10
    seen, todo = set(), deque([(start, 0, ())])
    while todo:
        q, i, st = todo.popleft()
        if (q, i, st) in seen: continue
        seen.add((q, i, st))
        if i == len(w) and q in accept: return True
        for (p, x, t, r, u) in moves:
            if p != q: continue
            if x and (i >= len(w) or w[i] != x): continue
            if t and (not st or st[-1] != t): continue
            st2 = st[:-1] if t else st
            for c in reversed(u): st2 = st2 + (c,)
            if len(st2) > max_stack: continue
            todo.append((r, i + (1 if x else 0), st2))
    return False

def grammar_to_pda(G, start, alphabet):
    moves = [('q0', '', '', 'q', start + '$')]                # push S then $: we push strings, see below
    for A, rs in G.items():
        for r in rs: moves.append(('q', '', A, 'q', r))
    for a in alphabet: moves.append(('q', a, a, 'q', ''))
    moves.append(('q', '', '$', 'f', ''))
    return moves

# our simulator pushes the string u so that its first symbol ends on top (u is pushed in reverse)
pd = grammar_to_pda(dyck, 'S', '()')
print("grammar -> PDA for Dyck words agrees on all words up to length 10:",
      all(pda_accepts(pd, 'q0', {'f'}, w) == is_dyck(w) for w in words('()', 10)))

pal_pda = [('s', '', '', 'p', '$')]
for c in 'ab':
    pal_pda += [('p', c, '', 'p', c), ('m', c, c, 'm', ''), ('p', c, '', 'm', '')]   # push, match, skip the middle letter
pal_pda += [('p', '', '', 'm', ''), ('m', '', '$', 'f', '')]
print("palindrome PDA agrees on all words up to length 12:",
      all(pda_accepts(pal_pda, 's', {'f'}, w) == (w == w[::-1]) for w in words('ab', 12)))
eqpda = [('q0', '', '', 'q', '$')] + [('q', c, d, 'q', '') for c, d in [('a', 'b'), ('b', 'a')]] + \
        [('q', c, '', 'q', c) for c in 'ab'] + [('q', '', '$', 'f', '')]
print("PDA of Exercise 2.9 agrees on all words up to length 12:",
      all(pda_accepts(eqpda, 'q0', {'f'}, w) == (w.count('a') == w.count('b')) for w in words('ab', 12)))

The simulator explores all the guesses of the automaton, so it is exponential in the worst case in principle, but the configurations are few here (a bounded stack is enough because a useful run never needs more stack than about twice the input). The palindrome automaton guesses the middle, possibly skipping a middle letter; the PDA of Exercise 2.9 may push a letter on top of the other one, a bad move that the invariant $s=d(u)$ makes harmless.

### P2.5 The complement of $\{ww\}$

In [ ]:
odd = {'A': ['aAa', 'aAb', 'bAa', 'bAb', 'a'], 'B': ['aBa', 'aBb', 'bBa', 'bBb', 'b'],
       'O': ['aE', 'bE'], 'E': ['aaE', 'abE', 'baE', 'bbE', ''], 'S': ['AB', 'BA', 'O']}
H = to_cnf(odd, 'S')
is_square = lambda w: len(w) % 2 == 0 and w[:len(w) // 2] == w[len(w) // 2:]
ws = [w for w in words('ab', 12) if w]
print(f"grammar of Exercise 2.7 = complement of {{ww}} on all {len(ws)} nonempty words of length <= 12:",
      all((cyk(H, 'S', w) > 0) == (not is_square(w)) for w in ws))

The grammar generates exactly the words that are not squares, as Exercise 2.7 proves, on all $8190$ nonempty words up to length $12$. The squares themselves cannot be generated by any grammar (Proposition 2.12): the context-free languages are not closed under complement, and this is an example in the other direction.